# 🥁 Drum Copy — Google Colab 実行ノートブック

楽曲ファイル（wav / mp3）からドラムパートを分離し、MIDIファイルとして出力します。

```
入力音声 → [Demucs: stem分離] → ドラムwav → [omnizart: 自動採譜] → MIDI
```

## 実行前の準備

1. **GPUランタイムを有効化**  
   「ランタイム」→「ランタイムのタイプを変更」→「ハードウェアアクセラレータ: GPU（T4）」

2. **GitHub から自動取得**  
   セル 4 を実行すると `https://github.com/Hukumaro/drum-copy` から自動的にクローン / pull されます

3. **入力音声を配置**  
   `マイドライブ/drum-copy_data/input/` に処理したい wav / mp3 ファイルを入れる

4. **上から順にセルを実行**（「ランタイム」→「すべてのセルを実行」でも可）

> 出力 MIDI は `マイドライブ/drum-copy_data/output/` に保存されます。

## 1. GPU 確認

In [1]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")
else:
    print("⚠️  GPU が検出されません。")
    print("「ランタイム」→「ランタイムのタイプを変更」→「ハードウェアアクセラレータ: GPU」を設定してください。")
    print("CPU でも実行できますが、処理に数十分かかる場合があります。")

✅ GPU: Tesla T4  (15.6 GB VRAM)


## 2. パッケージインストール

> ランタイムが変わるたびに再実行が必要です。初回は数分かかります。

In [ ]:
# omnizart は vamp に依存しているが Colab 環境では不要なためパッチインストール
import subprocess, sys, shutil, re, os

def pip(args: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args.split(), check=True)

def pip_soft(pkg: str) -> None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True)
    status = "✓" if r.returncode == 0 else "⚠️ (skip)"
    print(f"  {status} {pkg}")

print("Installing demucs...")
pip("demucs")

print("Installing soundfile...")
pip("soundfile")

print("Upgrading setuptools...")
pip("setuptools --upgrade")

print("Installing madmom...")
pip("madmom --no-build-isolation")

print("Cloning omnizart...")
omnizart_src = "/content/_omnizart_src"
if os.path.exists(omnizart_src):
    shutil.rmtree(omnizart_src)
subprocess.run(["git", "clone", "--depth=1",
                "https://github.com/Music-and-Culture-Technology-Lab/omnizart.git",
                omnizart_src], check=True, capture_output=True)

# pyproject.toml / setup.cfg を削除して build backend に依存しないようにする
for fname in ["pyproject.toml", "setup.cfg"]:
    p = os.path.join(omnizart_src, fname)
    if os.path.exists(p):
        os.remove(p)

# setup.py から vamp 依存を除去
setup_path = os.path.join(omnizart_src, "setup.py")
with open(setup_path, "r") as f:
    src = f.read()
src = re.sub(r"['\"]vamp[^'\"]*['\"],?\s*", "", src)
with open(setup_path, "w") as f:
    f.write(src)

# pip install は Python 3.12 環境で egg_info が失敗するため
# sys.path 追加 + 依存パッケージを個別インストールする方式に変更
print("Installing omnizart dependencies individually...")
for dep in [
    "tensorflow",
    "pretty_midi",
    "music21",
    "h5py",
    "ruamel.yaml",
    "coloredlogs",
    "click",
    "mir_eval",
    "pydantic",
]:
    pip_soft(dep)

if omnizart_src not in sys.path:
    sys.path.insert(0, omnizart_src)

# Python 3.12 removed pkgutil.ImpImporter; patch before importing omnizart
import pkgutil
if not hasattr(pkgutil, "ImpImporter"):
    class _ImpImporter:
        def __init__(self, path=None): self.path = path
        def find_module(self, fullname, path=None): return None
        def iter_modules(self, prefix=""): return iter([])
    pkgutil.ImpImporter = _ImpImporter
if not hasattr(pkgutil, "ImpLoader"):
    class _ImpLoader:
        pass
    pkgutil.ImpLoader = _ImpLoader

print("✅ インストール完了")

## 3. Google Drive マウント

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive マウント完了")

## 4. プロジェクト読み込み（GitHub から常に最新を取得）

GitHub リポジトリ `Hukumaro/drum-copy` から自動的に `git pull` または `git clone` を行い、
Google Drive 内のコードを常に最新状態に保ちます。  
`input/` / `output/` フォルダのユーザーファイルは保護されます。

In [ ]:
import sys, subprocess, shutil
from pathlib import Path

# ── Google Drive 上のプロジェクトルート ─────────────────────────────
# 配置場所を変えた場合はここを修正してください
PROJECT_ROOT = Path("/content/drive/MyDrive/drum-copy")
GITHUB_REPO  = "https://github.com/Hukumaro/drum-copy"
# ────────────────────────────────────────────────────────────────────

def _run(cmd, cwd=None):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result.stdout.strip()

def _backup_user_dirs():
    """input/ output/ をコンテンツ一時領域に退避する"""
    saved = {}
    for d in ["input", "output"]:
        src = PROJECT_ROOT / d
        if src.exists():
            tmp = Path(f"/content/_bak_{d}")
            if tmp.exists():
                shutil.rmtree(str(tmp))
            shutil.copytree(str(src), str(tmp))
            saved[d] = tmp
    return saved

def _restore_user_dirs(saved):
    for d, tmp in saved.items():
        dest = PROJECT_ROOT / d
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(tmp), str(dest), dirs_exist_ok=True)
        shutil.rmtree(str(tmp))

if (PROJECT_ROOT / ".git").exists():
    print(f"🔄 git pull — {GITHUB_REPO}")
    out = _run(["git", "pull", "--ff-only"], cwd=str(PROJECT_ROOT))
    print(out or "Already up to date.")
    print("✅ 最新コードに更新しました")
elif PROJECT_ROOT.exists():
    print("⚠️  .git が見つかりません。input/output を退避して再クローンします...")
    saved = _backup_user_dirs()
    shutil.rmtree(str(PROJECT_ROOT))
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print(_run(["git", "clone", GITHUB_REPO, str(PROJECT_ROOT)]))
    _restore_user_dirs(saved)
    print("✅ クローン完了（input/output は保持）")
else:
    print(f"📥 初回クローン — {GITHUB_REPO}")
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print(_run(["git", "clone", GITHUB_REPO, str(PROJECT_ROOT)]))
    print("✅ クローン完了")

sys.path.insert(0, str(PROJECT_ROOT))
%cd {PROJECT_ROOT}
print(f"\n✅ プロジェクトパス: {PROJECT_ROOT}")

## 5. パス設定

入力・出力ディレクトリは **コードとは別フォルダ** `drum-copy_data/` に保存されます。  
**通常はここを変更する必要はありません。**

| フォルダ | Google Drive 上のパス |
|---------|---------------------|
| 入力音声 | `マイドライブ/drum-copy_data/input/` |
| 出力 MIDI | `マイドライブ/drum-copy_data/output/` |
| コード | `マイドライブ/drum-copy/` |

In [ ]:
from backends.colab import ColabBackend

# ── 設定 ────────────────────────────────────────────────────────────
MODEL      = "htdemucs"    # Demucs モデル（htdemucs / htdemucs_ft / mdx_extra など）
OVERWRITE  = False         # True にすると既存の MIDI を上書き
# ────────────────────────────────────────────────────────────────────

# コードとは別フォルダにユーザーデータを保持する（案 B）
DATA_ROOT = Path("/content/drive/MyDrive/drum-copy_data")

backend = ColabBackend(
    data_root=str(DATA_ROOT),
    use_gdrive=False,  # Drive は既にマウント済み
    tmp_dir="/content/tmp/drum-copy",
)
backend.setup()
paths = backend.get_paths()

SUPPORTED = {".wav", ".mp3"}
audio_files = [p for p in sorted(paths.input_dir.iterdir()) if p.suffix.lower() in SUPPORTED]

print(f"入力ディレクトリ : {paths.input_dir}")
print(f"出力ディレクトリ : {paths.output_dir}")
print(f"一時ディレクトリ : {paths.tmp_dir}")
print(f"モデル          : {MODEL}")
print()
print(f"入力ファイル ({len(audio_files)} 件):")
for f in audio_files:
    print(f"  - {f.name}")
if not audio_files:
    print(f"⚠️  {paths.input_dir} に音声ファイルが見つかりません。")

### (オプション) Google Drive を使わず直接アップロードする場合

下のセルのコメントを外して実行すると、PCからファイルをアップロードできます。

In [ ]:
# from google.colab import files as colab_files
# import shutil
#
# uploaded = colab_files.upload()
# for fname in uploaded:
#     dest = paths.input_dir / fname
#     shutil.move(fname, dest)
#     print(f"保存: {dest}")
#
# # アップロード後に audio_files を更新
# audio_files = [p for p in sorted(paths.input_dir.iterdir()) if p.suffix.lower() in SUPPORTED]

## 6. パイプライン実行

- **Step 1** : Demucs でドラムトラックを分離 → `tmp/` に保存  
- **Step 2** : omnizart でドラムを採譜 → `output/` に MIDI 保存

> 1曲あたりの目安: GPU(T4) で 2〜5分、CPU では 20〜60分

In [ ]:
import logging
import shutil

from pipeline.stem_separation.separator import separate
from pipeline.transcription.transcriber import transcribe

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("drum_copy")

if not audio_files:
    raise FileNotFoundError(f"入力ファイルがありません: {paths.input_dir}")

results = []
for audio_path in audio_files:
    midi_out = paths.output_dir / (audio_path.stem + ".mid")

    if midi_out.exists() and not OVERWRITE:
        logger.info("スキップ（既存）: %s", midi_out.name)
        results.append((audio_path.name, str(midi_out), "SKIP"))
        continue

    try:
        logger.info("=== Step 1: Stem 分離 — %s ===", audio_path.name)
        drums_wav = separate(audio_path, paths.tmp_dir, model=MODEL)

        logger.info("=== Step 2: MIDI 変換 — %s ===", drums_wav.name)
        midi_path = transcribe(drums_wav, paths.output_dir)

        if midi_path.stem != audio_path.stem:
            renamed = paths.output_dir / (audio_path.stem + ".mid")
            midi_path.rename(renamed)
            midi_path = renamed

        stem_tmp = paths.tmp_dir / audio_path.stem
        if stem_tmp.exists():
            shutil.rmtree(stem_tmp)

        results.append((audio_path.name, str(midi_path), "OK"))
        logger.info("完了: %s", midi_path)

    except Exception as exc:
        logger.error("失敗 %s: %s", audio_path.name, exc)
        results.append((audio_path.name, "", f"ERROR: {exc}"))

print("\n─── 結果 ─────────────────────────────────────")
for src, dst, status in results:
    icon = "✅" if status in ("OK", "SKIP") else "❌"
    print(f"{icon}  {src}  →  {dst or status}")

## 7. MIDI ダウンロード（オプション）

出力 MIDI は Google Drive の `output/` に自動保存されています。  
Colab から直接ダウンロードしたい場合は下のセルを実行してください。

In [ ]:
from google.colab import files as colab_files

midi_files = list(paths.output_dir.glob("*.mid"))
if midi_files:
    for f in midi_files:
        print(f"ダウンロード中: {f.name}")
        colab_files.download(str(f))
else:
    print("出力 MIDI ファイルが見つかりません。")